In [ ]:

# 1. Import libraries

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.base import clone

RANDOM_STATE = 42


In [ ]:

# 2. Load dataset

here = os.getcwd()
csv_path = os.path.join(here, "Student_Mental_health.csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"Could not find {csv_path}. Put Student_Mental_health.csv in the same folder as this notebook."
    )

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:

# 3. Basic dataset information

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(i, col)

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMissing values:")
missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)
display(missing)


In [ ]:

# 4. Clean obvious text formatting for EDA

eda = df.copy()

for col in eda.select_dtypes(include="object").columns:
    eda[col] = eda[col].astype("string").str.strip()

# Target distribution
target_col = "Do you have Depression?"

print("Target values:")
display(eda[target_col].value_counts(dropna=False))

plt.figure(figsize=(6,4))
eda[target_col].value_counts(dropna=False).plot(kind="bar")
plt.title("Depression Target Distribution")
plt.xlabel("Depression")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

# 5. Numeric EDA

numeric_candidates = [c for c in ["Age"] if c in eda.columns]

for col in numeric_candidates:
    eda[col] = pd.to_numeric(eda[col], errors="coerce")

if numeric_candidates:
    display(eda[numeric_candidates].describe())

    for col in numeric_candidates:
        plt.figure(figsize=(7,4))
        plt.hist(eda[col].dropna(), bins=15)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()


In [ ]:

# 6. Categorical EDA

categorical_eda_cols = [
    "Choose your gender",
    "Course_grouped",
    "Your current year of Study",
    "Marital status",
    "Do you have Anxiety?",
    "Do you have Panic attack?",
    "Did you seek any specialist for a treatment?"
]

# Create a grouped course column only for EDA
course_col = "What is your course?"
if course_col in eda.columns:
    course_counts = eda[course_col].value_counts(dropna=False)
    common_courses = course_counts[course_counts >= 4].index
    eda["Course_grouped"] = eda[course_col].apply(
        lambda x: x if x in common_courses else "Other"
    )

for col in categorical_eda_cols:
    if col in eda.columns:
        print(f"\n{col}")
        display(eda[col].value_counts(dropna=False).head(15))

        plt.figure(figsize=(8,4))
        eda[col].value_counts(dropna=False).head(12).plot(kind="bar")
        plt.title(f"{col} Distribution")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


In [ ]:

# 7. Depression rate by important categorical variables

def depression_rate_table(data, col):
    temp = data[[col, target_col]].dropna().copy()
    temp[target_col] = temp[target_col].astype(str).str.strip().str.title()
    return (
        temp.groupby(col)[target_col]
        .apply(lambda s: (s == "Yes").mean() * 100)
        .sort_values(ascending=False)
        .to_frame("Depression_Rate_Percent")
    )

for col in [
    "Choose your gender",
    "Your current year of Study",
    "Marital status",
    "Do you have Anxiety?",
    "Do you have Panic attack?",
    "Did you seek any specialist for a treatment?"
]:
    if col in eda.columns:
        print(f"\nDepression rate by {col}")
        display(depression_rate_table(eda, col))

# Age by target
if "Age" in eda.columns:
    plt.figure(figsize=(7,4))
    for label, group in eda.groupby(target_col):
        plt.hist(group["Age"].dropna(), bins=12, alpha=0.5, label=str(label))
    plt.title("Age Distribution by Depression Status")
    plt.xlabel("Age")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:

# 8. Feature engineering functions

data = df.copy()

# Clean all text columns
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].astype("string").str.strip()

# Target
data[target_col] = data[target_col].astype("string").str.strip().str.title()
data = data[data[target_col].isin(["Yes", "No"])].copy()
y = data[target_col].map({"No": 0, "Yes": 1}).astype(int)

# Age
data["Age"] = pd.to_numeric(data["Age"], errors="coerce")

# CGPA -> numeric midpoint
def cgpa_to_numeric(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    nums = re.findall(r"\d+(?:\.\d+)?", s)
    if not nums:
        return np.nan
    nums = [float(x) for x in nums]
    return sum(nums) / len(nums)

data["CGPA_numeric"] = data["What is your CGPA?"].apply(cgpa_to_numeric)

# Study year -> ordinal value
study_year_map = {
    "First Year": 1,
    "Second Year": 2,
    "Third Year": 3,
    "Fourth Year": 4,
    "Fifth Year": 5
}
data["Study_Year_Ordinal"] = (
    data["Your current year of Study"]
    .astype("string")
    .str.strip()
    .str.title()
    .map(study_year_map)
)

# Binary yes/no helper
def yes_no_to_binary(series):
    cleaned = series.astype("string").str.strip().str.lower()
    return cleaned.map({"yes": 1, "no": 0})

for col in [
    "Do you have Anxiety?",
    "Do you have Panic attack?",
    "Did you seek any specialist for a treatment?"
]:
    data[col + "_binary"] = yes_no_to_binary(data[col])

# Rare-course grouping
course_col = "What is your course?"
course_counts = data[course_col].value_counts()
common_courses = course_counts[course_counts >= 4].index
data["Course_grouped"] = data[course_col].apply(
    lambda x: x if x in common_courses else "Other"
)

display(data.head())


In [ ]:

# 9. Engineered feature summary

feature_cols = [
    "Choose your gender",
    "Age",
    "Course_grouped",
    "Study_Year_Ordinal",
    "CGPA_numeric",
    "Marital status",
    "Do you have Anxiety?_binary",
    "Do you have Panic attack?_binary",
    "Did you seek any specialist for a treatment?_binary"
]

X = data[feature_cols].copy()

print("Final feature matrix shape:", X.shape)
print("\nFeatures:")
for c in X.columns:
    print("-", c)

print("\nMissing values after feature engineering:")
display(X.isna().sum().to_frame("Missing_Count"))


In [ ]:

# 10. Train-test split BEFORE model fitting

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining target balance:")
display(y_train.value_counts(normalize=True).rename("proportion"))

print("\nTest target balance:")
display(y_test.value_counts(normalize=True).rename("proportion"))


In [ ]:

# 11. Preprocessing pipeline

categorical_features = ["Choose your gender", "Course_grouped", "Marital status"]
numeric_features = [
    "Age",
    "Study_Year_Ordinal",
    "CGPA_numeric",
    "Do you have Anxiety?_binary",
    "Do you have Panic attack?_binary",
    "Did you seek any specialist for a treatment?_binary"
]

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_preprocessor, numeric_features),
    ("cat", categorical_preprocessor, categorical_features)
])


In [ ]:

# 12. Four models

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=5,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=4,
        random_state=RANDOM_STATE
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    "Naive Bayes": GaussianNB()
}

pipelines = {
    name: Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}


In [ ]:

# 13. 5-fold stratified cross-validation model comparison

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

results = []

for name, pipe in pipelines.items():
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "CV Accuracy Mean": scores["test_accuracy"].mean(),
        "CV Accuracy SD": scores["test_accuracy"].std(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "Train Accuracy": scores["train_accuracy"].mean(),
        "Train F1": scores["train_f1"].mean()
    })

results_df = pd.DataFrame(results).sort_values("CV F1", ascending=False)
display(results_df.round(3))


In [ ]:

# 14. Visual model comparison

plot_df = results_df.set_index("Model")[["CV Accuracy Mean", "CV Precision", "CV Recall", "CV F1"]]

ax = plot_df.plot(kind="bar", figsize=(10,5))
ax.set_title("Model Comparison — 5-Fold Cross-Validation")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
ax.set_ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:

# 15. Quantitative overfitting check

overfit_df = results_df[[
    "Model", "Train Accuracy", "CV Accuracy Mean",
    "Train F1", "CV F1"
]].copy()

overfit_df["Accuracy Gap"] = (
    overfit_df["Train Accuracy"] - overfit_df["CV Accuracy Mean"]
)
overfit_df["F1 Gap"] = (
    overfit_df["Train F1"] - overfit_df["CV F1"]
)

display(overfit_df.round(3))

print("\nRule of thumb for this notebook:")
print("A large positive Train-CV gap is a warning sign for overfitting.")


In [ ]:

# 16. Train each model and check held-out test performance

test_results = []
fitted_models = {}

for name, pipe in pipelines.items():
    fitted = clone(pipe)
    fitted.fit(X_train, y_train)
    pred = fitted.predict(X_test)

    fitted_models[name] = fitted

    test_results.append({
        "Model": name,
        "Test Accuracy": accuracy_score(y_test, pred),
        "Test Precision": precision_score(y_test, pred, zero_division=0),
        "Test Recall": recall_score(y_test, pred, zero_division=0),
        "Test F1": f1_score(y_test, pred, zero_division=0)
    })

test_results_df = pd.DataFrame(test_results).sort_values(
    "Test F1", ascending=False
)

display(test_results_df.round(3))


In [ ]:

# 17. Final model selection
# Primary criterion: CV F1, because accuracy alone can hide class imbalance.

best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

print("Best model based on mean CV F1:", best_model_name)

best_pred = best_model.predict(X_test)

print("\n=== Final Test Classification Report ===")
print(classification_report(
    y_test,
    best_pred,
    target_names=["No Depression", "Depression"],
    zero_division=0
))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, best_pred)
print(cm)


In [ ]:

# 18. Confusion matrix for the selected model

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Depression", "Depression"]
)
disp.plot()
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()


In [ ]:

# 19. Optional: compare train vs CV vs test accuracy

comparison = results_df[["Model", "Train Accuracy", "CV Accuracy Mean"]].copy()
test_acc_map = test_results_df.set_index("Model")["Test Accuracy"]
comparison["Test Accuracy"] = comparison["Model"].map(test_acc_map)

comparison = comparison.set_index("Model")

ax = comparison.plot(kind="bar", figsize=(10,5))
ax.set_title("Overfitting / Generalization Check")
ax.set_ylabel("Accuracy")
ax.set_xlabel("Model")
ax.set_ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

display(comparison.round(3))


In [ ]:

# 20. Save the main result tables

results_df.round(4).to_csv("model_cv_comparison.csv", index=False)
test_results_df.round(4).to_csv("model_test_comparison.csv", index=False)
print("Saved:")
print("- model_cv_comparison.csv")
print("- model_test_comparison.csv")
